In [1]:
%pip install langchain langchain-community langchainhub langchain-chroma langchain-openai langchain-text-splitters langchain-classic beautifulsoup4 yfinance gradio

In [2]:
import os
import pprint
import getpass
import bs4
from bs4 import BeautifulSoup
from urllib.request import Request, urlopen
from google.colab import userdata

In [3]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage
from langchain_community.document_loaders import WebBaseLoader
from langchain_classic.chains import create_retrieval_chain, create_history_aware_retriever
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

import gradio as gr

/tmp/ipykernel_3056/4123518759.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


In [7]:
# ---------------------------------------------------------
# 1. SETUP & DATA LOADING
# ---------------------------------------------------------
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

def get_sitemap(url):
    req = Request(
        url=url,
        headers={"User-Agent": "Mozilla/5.0"}
    )
    response = urlopen(req)
    xml = BeautifulSoup(
        response,
        "lxml-xml",
        from_encoding=response.info().get_param("charset")
    )
    return xml

def get_urls(xml, name=None, data=None, verbose=False):
    urls = []
    for url in xml.find_all("url"):
        if xml.find("loc"):
            loc = url.findNext("loc").text
            urls.append(loc)
    return urls

url = "https://zerodha.com/varsity/chapter-sitemap2.xml"
xml = get_sitemap(url)
urls = get_urls(xml, verbose=False)

docs = []
# Loading just the first 10 for speed; adjust as needed
for i, url in enumerate(urls[:10]):
    loader = WebBaseLoader(url)
    docs.extend(loader.load())
    if i % 10 == 0:
        print("Loaded document index:", i)

/tmp/ipykernel_3056/3091849184.py:23: DeprecationWarning: Call to deprecated method findNext. (Replaced by find_next) -- Deprecated since version 4.0.0.
  loc = url.findNext("loc").text


Loaded document index: 0


In [8]:
# ---------------------------------------------------------
# 2. VECTORSTORE & RETRIEVER
# ---------------------------------------------------------
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever()

print(f"Len docs: {len(docs)}, Len splits: {len(splits)}")

Len docs: 10, Len splits: 286


In [9]:
# ---------------------------------------------------------
# 3. RAG CHAIN SETUP
# ---------------------------------------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0) # Updated to newer, cheaper model

system_prompt = (
    "You are a financial assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "If the question is not clear ask follow up questions"
    "\n\n"
    "{context}"
)

# History Aware Retriever
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

history_aware_retriever = create_history_aware_retriever(llm, retriever, contextualize_q_prompt)

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [13]:
import gradio as gr
from langchain_core.messages import AIMessage, HumanMessage

def predict(message, history):
    """
    message: str
    history: list of dicts in OpenAI-style format when type='messages'
             e.g. [{"role":"user","content":"hi"}, {"role":"assistant","content":"hello"}]
    """
    history_for_llm = []
    for item in history:
        role = item.get("role")
        content = item.get("content", "")
        if role == "user":
            history_for_llm.append(HumanMessage(content=content))
        elif role == "assistant":
            history_for_llm.append(AIMessage(content=content))

    result = rag_chain.invoke({"input": message, "chat_history": history_for_llm})
    return result["answer"]

#with gr.Blocks(theme="soft") as demo:
with gr.Blocks() as demo:
    gr.Markdown("# DocumentQABot")

    #chatbot = gr.Chatbot(height=400, type="messages", allow_tags=False)
    chatbot = gr.Chatbot(height=400, allow_tags=False)
    msg = gr.Textbox(
        placeholder="Hi! I am your virtual assistant, how can I help you today?",
        container=False,
        scale=7,
    )

    with gr.Row():
        undo = gr.Button("Delete Previous")
        clear = gr.Button("Clear")

    chat = gr.ChatInterface(
        fn=predict,
        chatbot=chatbot,
        textbox=msg,
        examples=["What is index fund?", "Where to buy stocks?"],
        title=None,  # already using Markdown title
    )

    # Clear chat
    clear.click(lambda: [], None, chatbot)

    # Undo last turn (removes last 2 messages if present: user+assistant)
    def undo_last(history):
        if not history:
            return []
        # In messages format, history is a list of dicts, typically alternating user/assistant
        return history[:-2] if len(history) >= 2 else []

    undo.click(undo_last, chatbot, chatbot)

#demo.launch(share=True, debug=False)
demo.launch(
    share=True,
    debug=False,
    theme=gr.themes.Soft()
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e065a323a443980c5d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [19]:
import gradio as gr
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage
from langchain_community.tools.yahoo_finance_news import YahooFinanceNewsTool

tools = [YahooFinanceNewsTool()]
agent_test_prompt = "What is the latest news about Indian stock market like infosys"

# LangGraph ReAct agent
#agent = create_react_agent(llm, tools)
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools
)

print("\n--- Running Agent (LangGraph) ---")
result = agent.invoke({"messages": [HumanMessage(content=agent_test_prompt)]})

# The last assistant message content:
print(result["messages"][-1].content)


--- Running Agent (LangGraph) ---


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: NSE"}}}


I couldn't find any specific news related to Infosys or the Indian stock market at this moment. It's possible that there are no recent updates available. If you have any other specific companies or topics in mind, please let me know!


In [18]:

from langchain_core.tools import tool
from langchain.agents import create_agent
import yfinance as yf

@tool
def search_stock_news(company: str) -> str:
    """Search Yahoo Finance for recent company news."""
    try:
        articles = yf.Search(company, news_count=5).news

        if not articles:
            return f"No news found for {company}."

        results = []
        for article in articles:
            content = article.get("content", article)
            title = content.get("title", "Untitled")
            url = content.get("canonicalUrl", {}).get("url", "")
            results.append(f"{title}\n{url}")

        return "\n\n".join(results)

    except Exception as e:
        return f"News search failed: {e}"


agent = create_agent(
    model=llm,
    tools=[search_stock_news]
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "Find recent news about Claritev"}
    ]
})

print(result["messages"][-1].content)


Here are some recent news highlights about Claritev:

1. **Transparency-Based Pricing**: Claritev has introduced a transparency-based pricing model, aiming to turn complex public market data into actionable reimbursement intelligence.

2. **Stock Performance**: Claritev Corporation (CTEV) saw a surge of 5.2%, raising questions about the potential for further gains.

3. **Board Appointment**: Claritev has appointed Jennifer Main to its Board of Directors.

4. **Earnings Conference Call**: The company announced its Third Quarter 2026 Earnings Conference Call.

5. **Omnicell Earnings**: In related news, Omnicell (OMCL) reported Q3 earnings and revenues that topped estimates.

If you need more details on any specific news item, feel free to ask!


In [23]:
import gradio as gr
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage
from langchain_community.tools.yahoo_finance_news import YahooFinanceNewsTool

tools = [search_stock_news]
agent_test_prompt = "What is growth of Claritev"

# LangGraph ReAct agent
#agent = create_react_agent(llm, tools)
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools
)

print("\n--- Running Agent (LangGraph) ---")
result = agent.invoke({"messages": [HumanMessage(content=agent_test_prompt)]})

# The last assistant message content:
print(result["messages"][-1].content)


--- Running Agent (LangGraph) ---
Claritev Corporation (CTEV) has recently made headlines with several developments:

1. **Transparency-Based Pricing**: Claritev introduced a new pricing model that focuses on transparency, aiming to convert complex public market data into actionable reimbursement intelligence.

2. **Stock Performance**: The company's stock surged by 5.2%, which may indicate potential for further gains.

3. **Leadership Changes**: Claritev appointed Jennifer Main to its Board of Directors, which could influence future strategic decisions.

4. **Upcoming Earnings Call**: The company announced its Third Quarter 2026 Earnings Conference Call, suggesting that they are preparing to share financial results and insights with investors.

These developments suggest that Claritev is actively working on enhancing its market position and transparency, which could contribute to its growth.
